# ----------------------------------- Data cleaning -----------------------------------

####

# Project plan

#### Importing important packages

In [31]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import math as math
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

import pyodbc

######

## Connecting to sql server to import data

In [32]:
# Finding ODBC Driver

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [33]:
# ============================================================
# 1. SQL Server connection
# ============================================================

server = r"DESKTOP-0SIEUH6\SQLEXPRESS"
database = "LogisticsAnalyticsDB"

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# ============================================================
# 2. Get all tables from dbo schema
# ============================================================

tables_query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;
"""

tables = pd.read_sql(tables_query, conn)

print("Tables found:")
print(tables)

Tables found:
       TABLE_NAME
0    Dim_Carriers
1   Dim_Customers
2       Dim_Dates
3   Dim_Employees
4   Dim_Locations
5    Dim_Products
6      Dim_Routes
7    Dim_Vehicles
8  Dim_Warehouses
9  Fact_Shipments


####

### Importing datasets from sql server in form of tables

In [34]:
# Dictionary to hold all DataFrames
dataframes = {}

for table_name in tables["TABLE_NAME"]:
    
    query = f"SELECT * FROM dbo.[{table_name}]"
    
    dataframes[table_name] = pd.read_sql(query, conn)
    
    # Create a variable with the same name as the SQL table
    globals()[table_name] = dataframes[table_name]
    
    print(f"{table_name}: {dataframes[table_name].shape}")

Dim_Carriers: (10, 5)
Dim_Customers: (3000, 6)
Dim_Dates: (1096, 10)
Dim_Employees: (900, 5)
Dim_Locations: (25, 6)
Dim_Products: (800, 7)
Dim_Routes: (350, 6)
Dim_Vehicles: (500, 6)
Dim_Warehouses: (30, 6)
Fact_Shipments: (22110, 30)


####

### EDA starts here:

###### Getting the data on the level of analysis:

In [35]:
# ============================================================
# MERGE ALL TABLES INTO ONE MASTER DATAFRAME
# ============================================================

def merge_all_tables(
    Fact_Shipments,
    Dim_Dates,
    Dim_Customers,
    Dim_Products,
    Dim_Routes,
    Dim_Warehouses,
    Dim_Employees,
    Dim_Vehicles,
    Dim_Carriers,
    Dim_Locations
):

    df = Fact_Shipments.copy()

    # --------------------------------------------------------
    # Helper function
    # --------------------------------------------------------

    def merge_dimension(df, dimension, left_key, right_key):
        
        # Keep only columns that are not already in df
        # plus the right-side join key
        new_columns = [
            col
            for col in dimension.columns
            if col not in df.columns or col == right_key
        ]

        dimension = dimension[new_columns].copy()

        df = df.merge(
            dimension,
            left_on=left_key,
            right_on=right_key,
            how="left",
            validate="many_to_one"
        )

        # Remove duplicate right-side key
        if left_key != right_key:
            df.drop(columns=right_key, inplace=True)

        return df

    # --------------------------------------------------------
    # Direct Fact → Dimension merges
    # --------------------------------------------------------

    merges = [
        (Dim_Dates,      "DateKey",     "DateKey"),
        (Dim_Customers,  "CustomerID",  "CustomerID"),
        (Dim_Products,   "ProductID",   "ProductID"),
        (Dim_Routes,     "RouteID",     "RouteID"),
        (Dim_Warehouses, "WarehouseID", "WarehouseID"),
        (Dim_Employees,  "EmployeeID",  "EmployeeID"),
        (Dim_Vehicles,   "VehicleID",   "VehicleID"),
        (Dim_Carriers,   "CarrierID",   "CarrierID")
    ]

    for dimension, left_key, right_key in merges:

        df = merge_dimension(
            df,
            dimension,
            left_key,
            right_key
        )

    # --------------------------------------------------------
    # Customer → Location
    # --------------------------------------------------------

    customer_location = Dim_Locations.add_prefix(
        "CustomerLocation_"
    )

    df = merge_dimension(
        df,
        customer_location,
        "LocationID",
        "CustomerLocation_LocationID"
    )

    # --------------------------------------------------------
    # Route → Origin Location
    # --------------------------------------------------------

    origin_location = Dim_Locations.add_prefix(
        "OriginLocation_"
    )

    df = merge_dimension(
        df,
        origin_location,
        "OriginLocationID",
        "OriginLocation_LocationID"
    )

    # --------------------------------------------------------
    # Route → Destination Location
    # --------------------------------------------------------

    destination_location = Dim_Locations.add_prefix(
        "DestinationLocation_"
    )

    df = merge_dimension(
        df,
        destination_location,
        "DestinationLocationID",
        "DestinationLocation_LocationID"
    )

    return df

In [36]:
df = merge_all_tables(Fact_Shipments, Dim_Dates, Dim_Customers,
                     Dim_Products, Dim_Routes, Dim_Warehouses,
                     Dim_Employees, Dim_Vehicles, Dim_Carriers,
                     Dim_Locations)


df.head()

,ShipmentID,DateKey,CustomerID,ProductID,CarrierID,VehicleID,WarehouseID,EmployeeID,RouteID,ShipmentStatus,...,OriginLocation_City,OriginLocation_State,OriginLocation_Region,OriginLocation_Country,OriginLocation_Zone,DestinationLocation_City,DestinationLocation_State,DestinationLocation_Region,DestinationLocation_Country,DestinationLocation_Zone
0,1,20240816,1690,456,9,315,7,816.0,27,Delivered,...,Bengaluru,Karnataka,South,India,Zone-A,Nagpur,Maharashtra,West,India,Zone-C
1,2,20220209,2755,785,2,74,23,549.0,303,Delivered,...,Lucknow,Uttar Pradesh,North,India,Zone-A,Indore,Madhya Pradesh,Central,India,Zone-C
2,3,20221024,2601,376,10,56,22,676.0,147,Returned,...,Nashik,Maharashtra,West,India,Zone-A,Bengaluru,Karnataka,South,India,Zone-A
3,4,20241113,1104,198,5,53,10,553.0,177,Delivered,...,Coimbatore,Tamil Nadu,South,India,Zone-A,Visakhapatnam,Andhra Pradesh,South,India,Zone-A
4,5,20240607,2129,354,8,150,25,254.0,333,Delivered,...,Ludhiana,Punjab,North,India,Zone-C,Visakhapatnam,Andhra Pradesh,South,India,Zone-A


In [37]:
# Creating function for basic Exploratory data analysis

def eda(df):


    #pd.set_option("display.max_columns", None)   # show all columns
    #pd.set_option("display.max_rows", None)       # show all rows  <-- ADD THIS
    #pd.set_option("display.min_rows", None)        # don't collapse to a preview  <-- ADD THIS
    #pd.set_option("display.width", None)          # don't wrap based on terminal width
    #pd.set_option("display.max_colwidth", None)   # don't truncate long cell values (optional)

    print("=" * 100)
    print("📊 EDA SUMMARY")
    print("=" * 100)

    # =========================================================
    # 1. FIRST 5 ROWS
    # =========================================================

    print("\n1️⃣ FIRST 5 ROWS")
    display(df.head(5))

    # =========================================================
    # 2. DUPLICATE ROWS
    # =========================================================

    print("\n2️⃣ DUPLICATE ROWS")

    duplicate_count = df.duplicated().sum()
    duplicate_pct = (duplicate_count / len(df)) * 100

    print(f"Duplicate Rows : {duplicate_count:,}")
    print(f"Duplicate %    : {duplicate_pct:.2f}%")

    # =========================================================
    # 3. DATAFRAME INFO
    # =========================================================

    print("\n3️⃣ DATAFRAME INFO")
    df.info()

    # =========================================================
    # 4. NULL SUMMARY
    # =========================================================

    print("\n4️⃣ NULL SUMMARY")

    null_summary = pd.DataFrame({
        "Non_Null": df.notna().sum(),
        "Null_Count": df.isna().sum(),
        "Null_%": (df.isna().mean() * 100).round(2)
    })

    # Only show columns containing nulls
    null_summary = null_summary[
        null_summary["Null_Count"] > 0
    ].sort_values(
        by="Null_Count",
        ascending=False
    )

    if not null_summary.empty:
        display(null_summary)
    else:
        print("✅ No missing values found.")

    # =========================================================
    # 5. UNIQUE VALUES
    # =========================================================

    print("\n5️⃣ UNIQUE VALUES")

    unique_summary = pd.DataFrame({
        "Unique_Count": df.nunique(dropna=True),
        "Unique_%": (
            df.nunique(dropna=True) / len(df) * 100
        ).round(2)
    }).sort_values(
        by="Unique_Count",
        ascending=False
    )

    display(unique_summary)

    # =========================================================
    # 6. NUMERICAL STATISTICS
    # =========================================================

    print("\n6️⃣ NUMERICAL STATISTICS")

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    if len(numeric_cols) > 0:

        numeric_summary = df[numeric_cols].describe().T

        # Add median
        numeric_summary["median"] = (
            df[numeric_cols].median()
        )

        display(numeric_summary)

    else:
        print("No numerical columns found.")

    # =========================================================
    # 7. CATEGORICAL STATISTICS
    # =========================================================

    print("\n7️⃣ CATEGORICAL STATISTICS")

    categorical_cols = df.select_dtypes(
        include=["object", "category"]
    ).columns

    if len(categorical_cols) > 0:

        categorical_summary = pd.DataFrame({
            "Unique_Count": df[categorical_cols].nunique(
                dropna=True
            ),

            "Most_Common": [
                (
                    df[col]
                    .mode(dropna=True)
                    .iloc[0]
                    if not df[col]
                    .mode(dropna=True)
                    .empty
                    else None
                )
                for col in categorical_cols
            ],

            "Most_Common_Freq": [
                (
                    df[col]
                    .value_counts(dropna=True)
                    .iloc[0]
                    if not df[col]
                    .value_counts(dropna=True)
                    .empty
                    else 0
                )
                for col in categorical_cols
            ]
        }, index=categorical_cols)

        display(categorical_summary)

    else:
        print("No categorical columns found.")

    # =========================================================
    # 8. POTENTIAL CATEGORICAL VARIABLES DISGUISED AS NUMERIC
    # =========================================================

    print(
        "\n8️⃣ POTENTIAL CATEGORICAL VARIABLES "
        "DISGUISED AS NUMERIC"
    )

    candidates = []

    for col in numeric_cols:

        unique_count = df[col].nunique(dropna=True)
        unique_pct = (unique_count / len(df)) * 100

        # Flag low-cardinality numeric columns
        if unique_count <= 20 or unique_pct <= 1:

            candidates.append({
                "Column": col,
                "Unique_Count": unique_count,
                "Unique_%": round(unique_pct, 2),
                "Sample_Values": (
                    df[col]
                    .dropna()
                    .unique()[:10]
                    .tolist()
                )
            })

    if candidates:

        potential_categorical = pd.DataFrame(
            candidates
        ).sort_values(
            by="Unique_Count"
        )

        display(potential_categorical)

    else:
        print("✅ No obvious numeric columns found.")

    print("\n" + "=" * 100)
    print("✅ EDA COMPLETED")
    print("=" * 100)

In [38]:
eda(df)

📊 EDA SUMMARY

1️⃣ FIRST 5 ROWS


,ShipmentID,DateKey,CustomerID,ProductID,CarrierID,VehicleID,WarehouseID,EmployeeID,RouteID,ShipmentStatus,...,OriginLocation_City,OriginLocation_State,OriginLocation_Region,OriginLocation_Country,OriginLocation_Zone,DestinationLocation_City,DestinationLocation_State,DestinationLocation_Region,DestinationLocation_Country,DestinationLocation_Zone
0,1,20240816,1690,456,9,315,7,816.0,27,Delivered,...,Bengaluru,Karnataka,South,India,Zone-A,Nagpur,Maharashtra,West,India,Zone-C
1,2,20220209,2755,785,2,74,23,549.0,303,Delivered,...,Lucknow,Uttar Pradesh,North,India,Zone-A,Indore,Madhya Pradesh,Central,India,Zone-C
2,3,20221024,2601,376,10,56,22,676.0,147,Returned,...,Nashik,Maharashtra,West,India,Zone-A,Bengaluru,Karnataka,South,India,Zone-A
3,4,20241113,1104,198,5,53,10,553.0,177,Delivered,...,Coimbatore,Tamil Nadu,South,India,Zone-A,Visakhapatnam,Andhra Pradesh,South,India,Zone-A
4,5,20240607,2129,354,8,150,25,254.0,333,Delivered,...,Ludhiana,Punjab,North,India,Zone-C,Visakhapatnam,Andhra Pradesh,South,India,Zone-A



2️⃣ DUPLICATE ROWS
Duplicate Rows : 0
Duplicate %    : 0.00%

3️⃣ DATAFRAME INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 84 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ShipmentID                    22110 non-null  int64  
 1   DateKey                       22110 non-null  int64  
 2   CustomerID                    22110 non-null  int64  
 3   ProductID                     22110 non-null  int64  
 4   CarrierID                     22110 non-null  int64  
 5   VehicleID                     22110 non-null  int64  
 6   WarehouseID                   22110 non-null  int64  
 7   EmployeeID                    21888 non-null  float64
 8   RouteID                       22110 non-null  int64  
 9   ShipmentStatus                22110 non-null  object 
 10  DeliveryMode                  22110 non-null  object 
 11  ShippingPriority              22110 n

,Non_Null,Null_Count,Null_%
PackageVolumeM3,21444,666,3.01
FuelConsumptionLiters,21557,553,2.50
WarehouseProcessingTimeHours,21667,443,2.00
PackageWeightKg,21668,442,2.00
WeatherCondition,21778,332,1.50
ShippingCost,21780,330,1.49
EmployeeID,21888,222,1.00
Role,21888,222,1.00
ExperienceYears,21888,222,1.00
ShiftType,21888,222,1.00



5️⃣ UNIQUE VALUES


...


7️⃣ CATEGORICAL STATISTICS


,Unique_Count,Most_Common,Most_Common_Freq
ShipmentStatus,4,Delivered,19834
DeliveryMode,8,Road,14144
ShippingPriority,3,Standard,13134
PaymentMethod,3,Prepaid,10083
WeatherCondition,5,Clear,12034
TrafficCondition,6,Medium,8792
FullDate,1096,2022-09-14,37
MonthName,12,January,1965
DayOfWeek,7,Wednesday,3255
CustomerName,2996,Customer_02130,17



8️⃣ POTENTIAL CATEGORICAL VARIABLES DISGUISED AS NUMERIC


,Column,Unique_Count,Unique_%,Sample_Values
9,Year,3,0.01,"[2024, 2022, 2023]"
8,Quarter,4,0.02,"[3, 1, 4, 2]"
3,PromisedDeliveryDays,8,0.04,"[2, 3, 5, 1, 6, 4, 7, 8]"
5,NumPreviousDelaysCustomer,9,0.04,"[1, 2, 0, 3, 4, 5, 6, 7, 8]"
0,CarrierID,10,0.05,"[9, 2, 10, 5, 8, 3, 4, 1, 7, 6]"
16,HistoricOnTimeRate,10,0.05,"[0.75, 0.78, 0.85, 0.88, 0.93, 0.82, 0.95, 0.9..."
7,Month,12,0.05,"[8, 2, 10, 11, 6, 12, 7, 5, 1, 4]"
15,VehicleAgeYears,15,0.07,"[8, 9, 1, 2, 0, 4, 3, 7, 13, 6]"
4,ActualDeliveryDays,18,0.08,"[1, 2, 4, 6, 9, 5, 8, 3, 14, 7]"
2,OrderQuantity,24,0.11,"[21, 3, 5, 23, 11, 16, 13, 10, 20, 15]"



✅ EDA COMPLETED


In [39]:
# Correcting column names

df.columns = df.columns.str.lower()
df.columns

Index(['shipmentid', 'datekey', 'customerid', 'productid', 'carrierid',
       'vehicleid', 'warehouseid', 'employeeid', 'routeid', 'shipmentstatus',
       'deliverymode', 'shippingpriority', 'paymentmethod', 'weathercondition',
       'trafficcondition', 'orderquantity', 'packageweightkg',
       'packagevolumem3', 'distancekm', 'shippingcost', 'ordervalue',
       'handlingtimehours', 'loadingtimehours', 'warehouseprocessingtimehours',
       'fuelconsumptionliters', 'promiseddeliverydays', 'actualdeliverydays',
       'deliverytimehours', 'numpreviousdelayscustomer', 'late_delivery',
       'fulldate', 'day', 'month', 'monthname', 'quarter', 'year', 'dayofweek',
       'isweekend', 'isholiday', 'customername', 'segment', 'locationid',
       'registrationdate', 'customertier', 'productname', 'category',
       'subcategory', 'unitweightkg', 'unitvolumem3', 'fragileflag',
       'originlocationid', 'destinationlocationid', 'routetype',
       'avgtrafficlevel', 'warehousename', 'war

In [40]:
# Keeping only completed shipments for which the delivery outcome
# is available and Late_Delivery can be observed.

# Delivered → completed delivery; valid target.
# Returned  → shipment reached the delivery stage; retained.
# In Transit → outcome not yet known; exclude.
# Cancelled  → shipment was not completed; late delivery is not a meaningful outcome.

df = df[df.shipmentstatus.isin(["Delivered", "Returned"])].copy()

In [41]:
# COLUMNS TO DROP
# These columns identify individual entities or shipments but do not represent useful measurable information for the model. 
# Raw high-cardinality identifiers can encourage memorization rather than learning general patterns

# ShipmentID   → unique identifier for each shipment
# CustomerName → customer identity already represented through customer-level engineered features
# ProductName  → product identity already represented elsewhere
# CarrierName  → carrier identity already represented by CarrierID

drop_columns = list()
drop_columns += ["shipmentid", "carriername", "productname", "customername"]

In [42]:
# REDUNDANT DATE / CALENDAR COLUMNS

# These columns already contain useful date-related features, so no additional meaningful information can be extracted from them..

# DateKey  → shipment date key; not used as a raw number
# FullDate → original shipment date; extracted date features are used instead
# Day      → redundant because day-of-week information is already available
# MonthName → redundant because numeric Month is available

drop_columns += ["datekey", "fulldate", "day", "monthname"]

In [43]:
# RegistrationDate is not used directly. Instead, it is transformed into CustomerTenureDays, 
# which measures how long the customer had been registered at the time of shipment.

drop_columns += ["registrationDate"]

In [44]:
# Location IDs are removed where the corresponding descriptive. location information is already available after merging.

# OriginLocationID      → OriginLocation_City etc.
# DestinationLocationID → DestinationLocation_City etc.

drop_columns += ["originlocationid", "destinationlocationid"]

In [45]:
# Warehouse IDs are removed where the corresponding descriptive
# warehouse information is already available after merging.

# WarehouseID → WarehouseName etc.

drop_columns += ["warehouseid"]

In [46]:
# Country columns have very cardinality as 1 in the dataset and therefore provide little additional information.

drop_columns += [ "customerlocation_country", "originlocation_country", "destinationlocation_country" ]

In [47]:
# Potential data leakage candidate
# ActualDeliveryDays is known only after the shipment has been delivered.

# Since the model is supposed to predict whether a shipment will be late, using ActualDeliveryDays would 
# expose information from the outcome itself and create target leakage.

# I think "deliverytimehours" can also create data leakage
# "shipmentstatus" can also create data leakage

drop_columns += ["actualdeliverydays", "deliverytimehours","shipmentstatus"]

In [48]:
# Dropping below columns bcz of very high cardinality
# dropping "routeid" as well coz other location related data is already there 
# Dropping "locationid" as it is not as useful coz other reated or extracted details are already there in th data

drop_columns += ["customerid", "productid", "employeeid", "vehicleid", "routeid", "locationid"]

In [49]:
# Cameback back to this worksheet coz
# Even after droping so many columns, after one hot encoding there are almost 300 columns which is too much to handle by my PC
# So dropping more columns


# Reduceing location columns
# I think zone which contains meaningful geographic differences, i can keep others i can drop
# To keep:
# --> originlocation_zone
# --> destinationlocation_zone
# --> customerlocation_zone
# To drop:
# --> customerlocation_city
# --> customerlocation_state
# --> customerlocation_region
# --> originlocation_city
# --> originlocation_state
# --> originlocation_region
# --> destinationlocation_city
# --> destinationlocation_state
# --> destinationlocation_region
drop_columns += ["customerlocation_city","customerlocation_state","customerlocation_region","originlocation_city",
                 "originlocation_state","originlocation_region","destinationlocation_city","destinationlocation_state","destinationlocation_region"]


# Removeing derived/redundant columns
# To keep:
# --> isweekend
# --> isholiday
# To drop:
# --> month
# --> quarter
# --> year
# --> shipmentyear
# --> shipmentdayofweek
drop_columns += ["month", "quarter", "year", "shipmentyear", "shipmentdayofweek"]


# Dropping columns unitweightkg and unitvolumem3 coz we already have packageweightkg and packagevolumem3 
# To keep:
# --> packageweightkg
# --> packagevolumem3
# To drop:
# --> unitweightkg
# --> unitvolumem3
drop_columns += ["unitweightkg", "unitvolumem3"]


# Dropping for possible data leakage
# To drop:
# --> handlingtimehours
# --> loadingtimehours
# --> warehouseprocessingtimehours
drop_columns += ["handlingtimehours", "loadingtimehours", "warehouseprocessingtimehours"]

In [50]:
# CUSTOMER TENURE

# Calculate how long the customer had been registered when the shipment occurred.
# This uses the shipment date rather than today's date so that the feature represents information available at prediction time.

df["registrationdate"] = pd.to_datetime(df["registrationdate"])
df["fulldate"] = pd.to_datetime(df["fulldate"])
df["customertenureyears"] = (df.fulldate - df.registrationdate).dt.days
df["customertenureyears"] = (df["customertenureyears"]/365.25) 

# The useful information from "registrationdate" has already been
# extracted into relevant features, so the original column can be dropped.
drop_columns += ["registrationdate"]

In [51]:
# SHIPMENT DATE

# Extract useful calendar features from the shipment date.

# ShipmentYear      → year of shipment
# Month             → month of shipment

# IsWeekend, IsHoliday and DayOfWeek are already available in the dataset, so no additional calculation is required for them.

df["shipmentyear"] = df.fulldate.dt.year
df["month"] = df.fulldate.dt.month


# Rename dayofweek to shipmentdayofweek for better clarity and consistency
df.rename(columns={"dayofweek": "shipmentdayofweek"}, inplace=True)

In [52]:
# DATATYPE CONVERSIONS

# These variables represent categories rather than quantities. So, for further analysis we will do datatype conversion 

# Year → different calendar periods
# Month → calendar category
# DayOfWeek → weekday category
# DeliveryMode → delivery method
# ShippingPriority → priority category
# PaymentMethod → payment category
# WeatherCondition → weather category
# TrafficCondition → traffic category

# Converting them to "category" makes their intended semantic type explicit before encoding for machine learning.

categorical_columns = [ "shipmentyear", "month", "shipmentdayofweek", "deliverymode", "shippingpriority", 
                       "paymentmethod", "weathercondition", "trafficcondition" ]

for col in categorical_columns:
    df[col] = df[col].astype("object")

In [53]:
# CONTINUOUS VARIABLES

# Keep these as numerical variables because their magnitude and differences carry meaningful information.

# promiseddeliverydays       → numerical duration
# numpreviousdelayscustomer  → count
# historicontimerate         → rate
# vehicleageyears            → continuous age
# capacityunits              → numerical capacity
# distancekm                 → numerical distance
# orderquantity              → numerical quantity
# packageweightkg            → continuous measurement
# packagevolumem3            → continuous measurement
# shippingcost               → monetary value
# ordervalue                 → monetary value
# handlingtimehours          → duration
# loadingtimehours           → duration
# warehouseprocessingtimehours → duration
# fuelconsumptionliters      → continuous measurement
# customertenureyears        → continuous duration
# ------------------------------------------------------------

numeric_columns = [ "promiseddeliverydays", "numpreviousdelayscustomer", "vehicleageyears", "capacityunits", "distancekm", "historicontimerate",
                   "orderquantity", "packageweightkg", "packagevolumem3", "shippingcost", "ordervalue", "handlingtimehours", 
                   "loadingtimehours", "warehouseprocessingtimehours", "fuelconsumptionliters", "customertenureyears" ]

# Checking their datatypes for confirmination
df.loc[:,numeric_columns].dtypes

promiseddeliverydays              int64
numpreviousdelayscustomer         int64
vehicleageyears                   int64
capacityunits                     int64
distancekm                      float64
historicontimerate              float64
orderquantity                     int64
packageweightkg                 float64
packagevolumem3                 float64
shippingcost                    float64
ordervalue                      float64
handlingtimehours               float64
loadingtimehours                float64
warehouseprocessingtimehours    float64
fuelconsumptionliters           float64
customertenureyears             float64
dtype: object

In [54]:
# ============================================================
# 8. CHECK FOR POTENTIAL NUMERIC → CATEGORICAL VARIABLES
# ============================================================

# ------------------------------------------------------------
# Some numeric columns may have a small number of unique values
# but can still represent meaningful numerical quantities.
#
# Examples:
#
# promiseddeliverydays
# numpreviousdelayscustomer
#
# These should NOT automatically be converted to categorical
# simply because they have low cardinality.
# ------------------------------------------------------------

potential_categorical = []

for col in df.select_dtypes(include="number").columns:

    # Skip columns already defined as numerical variables ---> earlier
    if col in numeric_columns:
        continue

    unique_count = df[col].nunique(dropna=True)

    if unique_count <= 30:
        potential_categorical.append({
            "column": col,
            "unique_count": unique_count,
            "values": df[col].dropna().unique().tolist()
        })

potential_categorical = pd.DataFrame(potential_categorical)

display(potential_categorical)

,column,unique_count,values
0,carrierid,10,"[9, 2, 10, 5, 8, 3, 4, 1, 7, 6]"
1,warehouseid,30,"[7, 23, 22, 10, 25, 17, 21, 28, 9, 6, 24, 27, ..."
2,actualdeliverydays,18,"[1, 2, 4, 6, 9, 5, 8, 3, 7, 10, 13, 11, 12, 14..."
3,quarter,4,"[3, 1, 4, 2]"
4,year,3,"[2024, 2022, 2023]"
5,locationid,25,"[24, 19, 21, 14, 6, 20, 1, 17, 3, 5, 15, 16, 1..."
6,originlocationid,25,"[3, 10, 24, 18, 23, 22, 25, 15, 1, 12, 19, 2, ..."
7,destinationlocationid,25,"[12, 13, 3, 19, 4, 1, 14, 10, 9, 20, 6, 22, 21..."


In [55]:
# Data type conversion 

change_to_categorical = potential_categorical.column.to_list()

df.loc[:, change_to_categorical] = df.loc[:, change_to_categorical].astype("object")
df.loc[:, change_to_categorical].dtypes

carrierid                object
warehouseid              object
actualdeliverydays       object
quarter                  object
year                     object
locationid               object
originlocationid         object
destinationlocationid    object
dtype: object

In [56]:
# DROP UNNECESSARY COLUMNS
# Remove the raw / redundant columns identified above.

df.drop(columns=drop_columns,inplace=True,errors="ignore")

In [57]:
# Target (y-variable) for the classification model.

# 0 → On-time
# 1 → Late

# Convert to integer so that the target is represented as a binary numerical variable.

df.loc[:, "late_delivery"] = df.loc[:, "late_delivery"].astype(int)

In [58]:
# Move 'Late_Delivery' to the last column to better clarity

# - Keep all other columns in their existing order
# - Remove 'Late_Delivery' from its current position
# - Append 'Late_Delivery' at the end of the DataFrame
# ------------------------------------------------------------

df = df[[col for col in df.columns if col != 'late_delivery']+ ['late_delivery']]

df.columns

Index(['carrierid', 'deliverymode', 'shippingpriority', 'paymentmethod',
       'weathercondition', 'trafficcondition', 'orderquantity',
       'packageweightkg', 'packagevolumem3', 'distancekm', 'shippingcost',
       'ordervalue', 'fuelconsumptionliters', 'promiseddeliverydays',
       'numpreviousdelayscustomer', 'isweekend', 'isholiday', 'segment',
       'customertier', 'category', 'subcategory', 'fragileflag', 'routetype',
       'avgtrafficlevel', 'warehousename', 'warehousetype', 'capacityunits',
       'automationlevel', 'role', 'experienceyears', 'shifttype',
       'vehicletype', 'capacitykg', 'fueltype', 'vehicleageyears',
       'carriertype', 'servicelevel', 'historicontimerate',
       'customerlocation_zone', 'originlocation_zone',
       'destinationlocation_zone', 'customertenureyears', 'late_delivery'],
      dtype='object')

In [63]:
# Standardize categorical variables

# - Remove leading/trailing whitespaces
# - Standardize inconsistent capitalization
# - Convert abbreviated values to their standard category names

# Delivery Mode
# Convert all values to lowercase first, then map them to standardized category names.
df['deliverymode'] = (df['deliverymode'].str.strip().str.lower().map({'road':'Road','sea':'Sea','air':'Air','rail':'Rail'}))


# Customer Segment
# Standardize capitalization and ensure 'SME' remains in its conventional uppercase format.
df['segment'] = (df['segment'].str.strip().str.lower().map({'retail':'Retail','sme':'SME','wholesale':'Wholesale','enterprise':'Enterprise' }))


# Traffic Condition
# Standardize capitalization and convert the abbreviation'med' to the standard category 'Medium'.
df['trafficcondition'] = ( df['trafficcondition'].str.strip().str.lower().map({'high':'High','medium':'Medium','med':'Medium','low':'Low'}) )

In [64]:
# goal is "clean once, reuse many times without re-running the heavy notebook,"
# So saving cleaned dataset in a pickle so that can be reused many times

df.to_pickle("C:/Users/Naveen/Desktop/GitHub/TransGlobe_Business/Classification_model/df.pkl")

# ----------------------------------------- END -----------------------------------------